In [0]:
from pyspark.sql.functions import concat_ws, col, lit, current_timestamp, when, upper, trim, to_date, coalesce, split, datediff, date_format, row_number, initcap, lpad, length, concat
from pyspark.sql.window import Window

df_compras = spark.table("azdb_sales_store.linio.silver_compras")

df_detalles = spark.table("azdb_sales_store.linio.silver_detalles")

df_fact_compras = (
    df_compras.alias("a")
    .join(df_detalles.alias("b"), on="factura", how="inner")
    .select(
        col("a.periodo"),
        col("a.venta_id"),
        col("a.factura"),
        col("a.tipo_compra"),
        col("a.fecha_orden"),
        col("a.fecha_entrega"),
        col("a.fecha_envio"),
        col("a.estado"),
        col("a.cliente_id"),
        col("a.vendedor"),
        col("a.departamento"),
        col("a.metodo_pago"),
        col("a.grupo_dias_envio"),
        col("b.detalle_id"),
        col("b.producto_id"),
        col("b.unidades"),
        col("b.subtotal"),
        current_timestamp().alias("fecha_actualizacion")
    )
)

df_fact_compras.write.format("delta").mode("overwrite").partitionBy("periodo").saveAsTable("azdb_sales_store.linio.gold_fact_compras")
